<h1 style="font-size: 1.6rem; font-weight: bold">ITO5163: Introduction to Cybersecurity</h1>
<h1 style="font-size: 1.6rem; font-weight: bold">Module 4: Integrity Verification</h1>
<p style="margin-top: 5px; margin-bottom: 5px;">Monash University Australia</p>
<p style="margin-top: 5px; margin-bottom: 5px;">Jupyter Notebook by: Tristan Sim Yook Min</p>
References: Information Source from Monash Faculty of Information Technology

---

### **Hash Functions**

Encryption keeps a message **secret**, but when it reaches the recipient, two questions remain:

- **Integrity**: Is this exactly the message that was sent, or was it **changed** on the way?
- **Authentication**: Did it really come from the claimed **sender**, or is someone impersonating them?

**Hash functions** are a key tool for answering both.

#### **What is a Hash Function?**

A **hash function** $H$ takes a message $M$ of **any length** and produces a **fixed-size** output $h$, called the **hash value**, **digest** or **fingerprint**:

$$h = H(M)$$

```mermaid
flowchart TB
    M["Message or data block M<br>(any length, L bits)"] --> H[/"Hash function H"\] --> h["Hash value h<br>(fixed length)"]
```

A good hash function produces outputs that look **random** and are **evenly spread**. Changing even **one bit** of the input changes about **half the bits** of the output. This is the **avalanche effect**.

| Input    | SHA-256 Hash |
|----------|--------------|
| `Hello`  | `185f8db32271fe25f561a6fc938b2e264306ec304eda518007d1764826381969` |
| `hello`  | `2cf24dba5fb0a30e26e83b2ac5b9e29e1b161e5c1fa7425e73043362938b9824` |
| `Hello!` | `334d016f755cd6dc58c53a86e183882f8ec14f52fb05345887c8a5edd42c87b7` |

Changing just `H` to `h` changes **125 of the 256** output bits.

- **Uses of Hash Functions**:
  - Message authentication
  - Digital signatures
  - Password storage
  - Intrusion detection: detecting changes to system files
  - Virus detection: identifying known malware by its hash
  - File integrity checks for downloads

#### **Cryptographic Hash Functions**

For cryptography, a hash function must be **cryptographically secure**. Two key requirements:

- **One-way**: given a hash value $h$, it's infeasible to find **any** message that hashes to $h$
- **Collision-resistant**: it's infeasible to find **two different messages** with the same hash

Because hashes are a fixed size but inputs can be any size, **collisions must exist**: there are far more possible messages than possible hashes. A secure hash function doesn't prevent collisions. It makes them **infeasible to find**. If anyone can find collisions in practice, the function is **broken**.

#### **A Simple XOR Hash Function**

A simple way to build an $n$-bit hash is to split the message into $n$-bit blocks and **XOR** them together:

$$C_i = b_{i1} \oplus b_{i2} \oplus \cdots \oplus b_{im}$$

- $C_i$: the $i$-th bit of the hash
- $m$: the number of $n$-bit blocks in the message
- $b_{ij}$: the $i$-th bit of the $j$-th block

This is called a **longitudinal redundancy check**. It works for **random data**, where every $n$-bit hash value is equally likely. It's weaker for **predictable data**: in normal text, the top bit of every character is always 0, so some hash bits are always 0 too.

An improvement is to **rotate** the running hash by 1 bit before XORing in each block. This mixes the bits more and makes the hash depend on **block order**.

#### **XOR Hash Execution for "Hello!" with 16-bit Blocks**

| Block | Characters | Bits |
|-------|------------|------|
| 1 | `He` | `0100100001100101` |
| 2 | `ll` | `0110110001101100` |
| 3 | `o!` | `0110111100100001` |

**Rotate-then-XOR (rotating left by 1 bit each step):**

| Step | Rotated Hash | XOR With Block | New Hash |
|------|--------------|----------------|----------|
| 1 | `0000000000000000` | `0100100001100101` | `0100100001100101` |
| 2 | `1001000011001010` | `0110110001101100` | `1111110010100110` |
| 3 | `1111100101001101` | `0110111100100001` | `1001011001101100` |

| Method | Hash of `He ll o!` | Hash of `ll He o!` (blocks swapped) |
|--------|--------------------|-------------------------------------|
| Simple XOR | `0100101100101000` | `0100101100101000` (same!) |
| Rotate-then-XOR | `1001011001101100` | `0100111001011010` (different) |

Final result:
- Simple XOR ignores block order, so swapping blocks gives a **collision**
- Rotating fixes that, but both are still **insecure**: an attacker can write any message, then **add one extra block** chosen to make the hash match the original

#### **Security Requirements of a Hash Function**

| Requirement | Meaning |
|-------------|---------|
| **Variable input size** | $H$ works on input of any length |
| **Fixed output size** | $H$ always produces the same length of output |
| **Efficiency** | $H(x)$ is quick to compute in software and hardware |
| **Preimage resistance** (one-way) | Given $h$, it's infeasible to find any $y$ with $H(y) = h$ |
| **Second preimage resistance** (weak collision resistance) | Given $x$, it's infeasible to find $y \neq x$ with $H(y) = H(x)$ |
| **Collision resistance** (strong collision resistance) | It's infeasible to find **any** pair $x \neq y$ with $H(x) = H(y)$ |
| **Pseudorandomness** | The output passes standard tests for randomness |

The input $x$ is called the **preimage** of $h = H(x)$.

- **Second Preimage vs Collision**:
  - **Second preimage**: the attacker is **given** a specific $x$ and must match it
  - **Collision**: the attacker can choose **both** messages freely, which is easier
  - Collisions can be found in about $2^{n/2}$ tries for an $n$-bit hash, by the **birthday paradox**, so hashes need to be twice as long as the security level wanted

#### **How the Requirements Relate**

```mermaid
flowchart LR
    CR["Collision resistant"] -- "always implies" --> SPR["Second preimage resistant"]
    PR["Preimage resistant<br>(one-way)"] -. "separate property" .- SPR
```

- **Collision resistance implies second preimage resistance**: if you can't find **any** colliding pair, you can't find one that matches a **given** message either
- **Preimage resistance is separate**: a function can have one without the other

| Hash Function | Output Size | Status |
|---------------|-------------|--------|
| MD5           | 128 bits    | **Broken**: collisions found in seconds |
| SHA-1         | 160 bits    | **Broken**: practical collision shown in 2017 |
| SHA-256 (SHA-2) | 256 bits  | Secure, widely used |
| SHA-3         | 224 to 512 bits | Secure, different internal design |

#### **Using Hash Functions for Message Authentication**

Here $\parallel$ means joining two values together (concatenation), $K$ is a symmetric encryption key, and $S$ is a shared secret value.

**(a) Encrypt the message and its hash together**

```mermaid
flowchart LR
    M["M"] --> C(("||"))
    M --> H["H"] --> C
    C --> E["E with K"] --> T["E(K, [M || H(M)])"] --> D["D with K"] --> R["M and H(M)"]
    R --> H2["Hash M"] --> CMP{"Compare"}
    R --> CMP
```

- The sender hashes $M$, joins the hash to $M$, and encrypts both
- The receiver decrypts, hashes $M$ again, and compares it with the hash that was sent
- Provides **confidentiality** and **authentication**

**(b) Encrypt only the hash**

```mermaid
flowchart LR
    M["M"] --> C(("||"))
    M --> H["H"] --> E["E with K"] --> C
    C --> T["M || E(K, H(M))"]
    T --> H2["Hash M"] --> CMP{"Compare"}
    T --> D["D with K"] --> CMP
```

- Only the hash is encrypted, and the message is sent as plain text
- The receiver can read $M$ directly, decrypts the hash, and compares it with their own hash of $M$
- Provides **authentication** only, with less encryption work

**(c) Hash with a shared secret, no encryption**

```mermaid
flowchart LR
    M["M"] --> C(("||"))
    M --> C2(("||"))
    S(["S"]) --> C2 --> H["H"] --> C
    C --> T["M || H(M || S)"]
    T --> C3["Join M with S<br>and hash"] --> CMP{"Compare"}
    T --> CMP
```

- Both sides share a secret value $S$, which is **never sent**
- The sender sends $M$ with $H(M \parallel S)$
- The receiver computes $H(M \parallel S)$ and compares
- An attacker who changes $M$ can't create the matching hash without knowing $S$
- Provides **authentication** without any encryption

**(d) Hash with a shared secret, then encrypt**

```mermaid
flowchart LR
    M["M"] --> C(("||"))
    M --> C2(("||"))
    S(["S"]) --> C2 --> H["H"] --> C
    C --> E["E with K"] --> T["E(K, [M || H(M || S)])"] --> D["D with K"] --> R["M and H(M || S)"]
    R --> C3["Join M with S<br>and hash"] --> CMP{"Compare"}
    R --> CMP
```

- Same as (c), but everything is encrypted with $K$
- Provides **confidentiality** and **authentication**

| Method | Sent | Confidentiality | Authentication | Needs |
|--------|------|-----------------|----------------|-------|
| (a) | $E(K, [M \parallel H(M)])$ | Yes | Yes | Key $K$ |
| (b) | $M \parallel E(K, H(M))$ | No | Yes | Key $K$ |
| (c) | $M \parallel H(M \parallel S)$ | No | Yes | Secret $S$ |
| (d) | $E(K, [M \parallel H(M \parallel S)])$ | Yes | Yes | Key $K$ and secret $S$ |

In practice, the standard way to combine a hash with a secret key is **HMAC**, which is designed to avoid weaknesses in simply joining the secret and message together.

<br>

***

### **Secure Hash Algorithm (SHA)**

The **Secure Hash Algorithm (SHA)** family is among the most widely used hash functions. The algorithms were designed by the **NSA** and published as standards by **NIST**.

| Version | Year | Status |
|---------|------|--------|
| **SHA-0** | 1993 | Withdrawn soon after release because of a weakness |
| **SHA-1** | 1995 | **Broken**: a real collision was found in 2017, and NIST is phasing it out by 2030 |
| **SHA-2** (SHA-224, SHA-256, SHA-384, SHA-512) | 2001 | Secure. **SHA-256** is the most widely used |
| **SHA-3** | 2015 | Secure. Based on a **completely different design** (Keccak) |

#### **Comparison of SHA Algorithms**

All sizes are in **bits**.

| Algorithm | Digest Size | Max Message Size | Block Size | Word Size | Steps | Collision Security |
|-----------|-------------|------------------|------------|-----------|-------|--------------------|
| SHA-1     | 160         | $< 2^{64}$       | 512        | 32        | 80    | Broken             |
| SHA-224   | 224         | $< 2^{64}$       | 512        | 32        | 64    | 112                |
| SHA-256   | 256         | $< 2^{64}$       | 512        | 32        | 64    | 128                |
| SHA-384   | 384         | $< 2^{128}$      | 1024       | 64        | 80    | 192                |
| SHA-512   | 512         | $< 2^{128}$      | 1024       | 64        | 80    | 256                |

Collision security is **half the digest size**, because of the birthday attack from lesson 10.1.

| Algorithm | Hash of `hello world` |
|-----------|-----------------------|
| SHA-1     | `2aae6c35c94fcfb415dbe95f408b9ce91ee846ed` |
| SHA-256   | `b94d27b9934d3e08a52e52d7da7dabfac484efe37a5380ee9088f7ace2efcde9` |
| SHA-3-256 | `644bcc7e564373040999aac89e7622f3ca71fba1d972fd94a31c3bfbf24e3938` |

#### **Inner Workings of SHA-512**

SHA-512 takes a message of **less than $2^{128}$ bits** and produces a **512-bit** digest. It processes the message in **1024-bit blocks**, one at a time. Each block is combined with the result of the previous block, so the final digest depends on **every** block.

```mermaid
flowchart LR
    IV(["H₀ = IV"]) --> F1["F<br>block M₁"] -- "H₁" --> F2["F<br>block M₂"] -- "H₂" --> D["..."] --> FN["F<br>block M_N"] -- "H_N" --> MD["Message digest<br>(512 bits)"]
```

#### **Step 1: Message Padding**

The message is padded so its length is **congruent to 896 mod 1024**, leaving exactly 128 bits free in the last block.

- **Padding Rules**:
  - Add a single **1** bit, followed by as many **0** bits as needed
  - Padding is **always** added, even if the message is already 896 mod 1024. In that case, a whole extra block of padding is added

#### **Step 2: Append the Length**

The original message length is added as a **128-bit unsigned integer**. The total length is now a **multiple of 1024**. Including the length makes it harder to find two different messages that hash to the same value.

| Message Length $L$ | Padding Bits ($1$ + zeros) | Length Field | Total | Blocks |
|--------------------|----------------------------|--------------|-------|--------|
| 24 (`abc`)         | 1 + 871 = 872              | 128          | 1024  | 1      |
| 896                | 1 + 1023 = 1024            | 128          | 2048  | 2      |
| 1000               | 1 + 919 = 920              | 128          | 2048  | 2      |

#### **Step 3: Initialise the Hash Buffer**

SHA-512 uses **eight 64-bit registers** $a, b, c, d, e, f, g, h$, a 512-bit buffer that holds intermediate and final results. Their starting values (the **IV**) are the first 64 bits of the fractional parts of the **square roots of the first eight primes** (2, 3, 5, 7, 11, 13, 17, 19).

| Register | Initial Value (hex) | From |
|----------|---------------------|------|
| a | `6A09E667F3BCC908` | $\sqrt{2}$ |
| b | `BB67AE8584CAA73B` | $\sqrt{3}$ |
| c | `3C6EF372FE94F82B` | $\sqrt{5}$ |
| d | `A54FF53A5F1D36F1` | $\sqrt{7}$ |
| e | `510E527FADE682D1` | $\sqrt{11}$ |
| f | `9B05688C2B3E6C1F` | $\sqrt{13}$ |
| g | `1F83D9ABFB41BD6B` | $\sqrt{17}$ |
| h | `5BE0CD19137E2179` | $\sqrt{19}$ |

These are "**nothing up my sleeve**" numbers: values anyone can check, which shows the designers didn't hide a weakness in the constants.

#### **Step 4: Process Each Block (80 Rounds)**

```mermaid
flowchart TB
    H0["Hᵢ₋₁ (512 bits)<br>= registers a to h"] --> R0["Round 0"]
    Mi["Block Mᵢ (1024 bits)"] --> WS["Message schedule<br>W₀ ... W₇₉ (64 bits each)"]
    WS -- "W₀" --> R0
    K(["Constants K₀ ... K₇₉"]) -- "K₀" --> R0
    R0 --> R1["Round 1"] --> RD["..."] --> R79["Round 79"]
    WS -- "W₇₉" --> R79
    K -- "K₇₉" --> R79
    R79 --> ADD(("+ mod 2⁶⁴<br>word by word"))
    H0 --> ADD
    ADD --> Hi["Hᵢ (512 bits)"]
```

- **The Compression Function F**:
  - Runs **80 rounds** on each 1024-bit block
  - Each round takes the 512-bit buffer ($a$ to $h$) as input and updates it
  - **$W_t$**: a 64-bit word derived from the current block. The first 16 come straight from the block, and the rest are mixed from earlier words
  - **$K_t$**: a 64-bit constant, from the first 64 bits of the fractional parts of the **cube roots of the first 80 primes**
  - $W_t$ and $K_t$ add **randomness**, so patterns in the plaintext don't show up in the digest

#### **Step 5: Add to the Previous Hash**

After round 79, the output is **added** to the buffer's value from **before round 0** ($H_{i-1}$). Each of the eight 64-bit words is added separately, **mod $2^{64}$**. This gives the intermediate hash $H_i$ for that block.

#### **Step 6: Final Message Digest**

After all $N$ blocks are processed, the final buffer is the digest:

$$H_0 = IV$$

$$H_i = \text{SUM}_{64}(H_{i-1},\ abcdefgh_i)$$

$$MD = H_N$$

- $IV$: the initial value of the $abcdefgh$ buffer
- $abcdefgh_i$: the output of the last round for block $i$
- $N$: the number of blocks in the message
- $\text{SUM}_{64}$: word-by-word addition mod $2^{64}$
- $MD$: the final message digest

For example, SHA-512 of `abc` (one block) is:

`ddaf35a193617abacc417349ae20413112e6fa4e89a97ea20a9eeee64b55d39a2192992a274fc1a836ba3c23a3feebbd454d4423643ce80e2a9ac94fa54ca49f`

#### **Why SHA-3?**

SHA-1 and SHA-2 share a **similar structure**, so researchers worried that an attack on SHA-1 might also work on SHA-2. NIST therefore required SHA-3 to be **fundamentally different**, so it could replace SHA-2 if it were ever broken.

| Feature | SHA-2 | SHA-3 |
|---------|-------|-------|
| Design | Merkle-Damgård: blocks chained through a compression function | **Sponge**: absorbs input, then "squeezes" out the output |
| Based on | Same family as SHA-1 | Keccak, chosen in a public competition |
| Status | Secure, widely used | Secure, a backup with a different design |


In [ ]:
# SHA hashing in Python.
# Part 1: the GeeksforGeeks Java example, using Python's built-in hashlib.
# Part 2: SHA-512 written from scratch, following the six steps in the notes.
import hashlib
import os


# ---------- Part 1: hashlib (what you would use in practice) ----------
def sha_hex(text, algorithm="sha1"):
    """Hash a string and return the digest as hex. Hashing is not encryption: it can't be reversed."""
    return hashlib.new(algorithm, text.encode("utf-8")).hexdigest()


print("Hash codes for 'GeeksForGeeks' and 'hello world':")
for algorithm in ["sha1", "sha256", "sha512", "sha3_256"]:
    for s in ["GeeksForGeeks", "hello world"]:
        h = sha_hex(s, algorithm)
        print(f"  {algorithm:8} {s:14} {h[:40]}{'...' if len(h) > 40 else ''}  ({len(h) * 4} bits)")


# ---------- Part 2: SHA-512 from scratch ----------
MASK = 2**64 - 1


def first_primes(count):
    primes, n = [], 2
    while len(primes) < count:
        if all(n % p for p in primes if p * p <= n):
            primes.append(n)
        n += 1
    return primes


def frac_bits_of_root(p, k):
    """First 64 bits of the fractional part of the k-th root of p, using exact integer maths."""
    x = int(round((p << (64 * k)) ** (1 / k)))          # starting guess
    while x**k > p << (64 * k):
        x -= 1
    while (x + 1) ** k <= p << (64 * k):
        x += 1
    return x & MASK                                       # drop the whole-number part


PRIMES = first_primes(80)
IV = [frac_bits_of_root(p, 2) for p in PRIMES[:8]]       # Step 3: square roots of the first 8 primes
K = [frac_bits_of_root(p, 3) for p in PRIMES]             # Step 4: cube roots of the first 80 primes


def rotr(x, n):
    return ((x >> n) | (x << (64 - n))) & MASK


def pad(message):
    """Steps 1 and 2: append a 1 bit, 0 bits up to 896 mod 1024, then the 128-bit length."""
    length = len(message) * 8
    padded = message + b"\x80"                            # the 1 bit (followed by seven 0 bits)
    while (len(padded) * 8) % 1024 != 896:
        padded += b"\x00"
    return padded + length.to_bytes(16, "big")


def compress(H, block):
    """Step 4 and 5: 80 rounds on one 1024-bit block, then add to the previous hash."""
    W = [int.from_bytes(block[8 * t:8 * t + 8], "big") for t in range(16)]
    for t in range(16, 80):                               # message schedule W16 ... W79
        s0 = rotr(W[t - 15], 1) ^ rotr(W[t - 15], 8) ^ (W[t - 15] >> 7)
        s1 = rotr(W[t - 2], 19) ^ rotr(W[t - 2], 61) ^ (W[t - 2] >> 6)
        W.append((W[t - 16] + s0 + W[t - 7] + s1) & MASK)

    a, b, c, d, e, f, g, h = H
    for t in range(80):                                   # the 80 rounds
        S1 = rotr(e, 14) ^ rotr(e, 18) ^ rotr(e, 41)
        ch = (e & f) ^ (~e & g)
        T1 = (h + S1 + ch + K[t] + W[t]) & MASK
        S0 = rotr(a, 28) ^ rotr(a, 34) ^ rotr(a, 39)
        maj = (a & b) ^ (a & c) ^ (b & c)
        T2 = (S0 + maj) & MASK
        a, b, c, d, e, f, g, h = (T1 + T2) & MASK, a, b, c, (d + T1) & MASK, e, f, g

    return [(x + y) & MASK for x, y in zip(H, [a, b, c, d, e, f, g, h])]   # SUM64


def sha512(message, trace=False):
    padded = pad(message)
    H = IV[:]                                             # H0 = IV
    blocks = [padded[i:i + 128] for i in range(0, len(padded), 128)]
    if trace:
        print(f"  {len(message) * 8} bits -> padded to {len(padded) * 8} bits = {len(blocks)} block(s)")
    for block in blocks:                                  # Hi = SUM64(Hi-1, abcdefgh_i)
        H = compress(H, block)
    return b"".join(x.to_bytes(8, "big") for x in H).hex()   # Step 6: MD = HN


print("\nSHA-512 constants:")
print("  IV a =", format(IV[0], "016X"), " (from sqrt 2)")
print("  K0   =", format(K[0], "016X"), " (from cube root of 2)")

print("\nPadding:")
for length_bytes in [3, 112, 125]:
    print(f"  {length_bytes * 8} bits -> {len(pad(b'x' * length_bytes)) * 8} bits")

print("\nSHA-512 of 'abc':")
mine = sha512(b"abc", trace=True)
print("  From scratch:", mine)
print("  hashlib:     ", hashlib.sha512(b"abc").hexdigest())
print("  Match:", mine == hashlib.sha512(b"abc").hexdigest())

# Check many random messages, including ones that need 2 or more blocks
ok = all(sha512(m) == hashlib.sha512(m).hexdigest()
         for m in (os.urandom(n) for n in range(0, 300, 7)))
print("\nMatches hashlib on messages from 0 to 293 bytes:", ok)

# Avalanche effect: change one character
h1, h2 = sha512(b"hello world"), sha512(b"hello worle")
diff = bin(int(h1, 16) ^ int(h2, 16)).count("1")
print(f"Avalanche: 'hello world' vs 'hello worle' differ in {diff} of 512 bits")


<br>

***

### **Message Digest Algorithm 5 (MD5)**

**MD5** was designed by **Ron Rivest** (the "R" in RSA) in **1991**. It's the fifth in his "Message Digest" series. Earlier versions weren't secure enough: MD2 and MD4 were published, MD4 was quickly found to be weak, and MD5 replaced it.

MD5 works in a **similar way to SHA**. It came first, and SHA-1 borrowed parts of its design from MD4 and MD5. SHA is **more secure**, but MD5 is **faster**.

| Feature | MD5 | SHA-1 | SHA-512 |
|---------|-----|-------|---------|
| Digest size | **128** bits | 160 bits | 512 bits |
| Block size | 512 bits | 512 bits | 1024 bits |
| Word size | 32 bits | 32 bits | 64 bits |
| Steps per block | 64 (4 rounds × 16) | 80 | 80 |
| Padding target | 448 mod 512 | 448 mod 512 | 896 mod 1024 |
| Length field | 64 bits | 64 bits | 128 bits |
| Status | **Broken** | **Broken** | Secure |

#### **How MD5 Works**

Each 512-bit block is split into **16 words of 32 bits**. Like SHA, MD5 pads the message, appends its length, splits it into blocks, and runs every block through several rounds that update a buffer.

```mermaid
flowchart LR
    IN(["Buffer a, b, c, d<br>(128 bits)"]) --> R1["Round 1<br>16 steps using F"] --> R2["Round 2<br>16 steps using G"] --> R3["Round 3<br>16 steps using H"] --> R4["Round 4<br>16 steps using I"] --> ADD(("+ mod 2³²"))
    IN --> ADD
    BLK["512-bit block<br>(16 words X₀ ... X₁₅)"] -.-> R1 & R2 & R3 & R4
    ADD --> OUT(["New buffer a, b, c, d"])
```

#### **Step 1: Padding**

The message is padded so its length is **congruent to 448 mod 512**. Like SHA, the padding is a **1** bit followed by **0** bits, and it's **always** added, even when the length already fits.

#### **Step 2: Append the Length**

The original length is added as a **64-bit unsigned integer**, so the total length is a **multiple of 512 bits**. Unlike SHA, MD5 writes it **low-order byte first** (little-endian).

| Message Length | Padding Bits | Length Field | Total | Blocks |
|----------------|--------------|--------------|-------|--------|
| 24 bits (`abc`) | 1 + 423 = 424 | 64 | 512 | 1 |
| 448 bits | 1 + 511 = 512 | 64 | 1024 | 2 |
| 480 bits | 1 + 479 = 480 | 64 | 1024 | 2 |

#### **Step 3: Initialise the Buffer**

MD5 uses a **four-word buffer** $(a, b, c, d)$. Each word is **32 bits**, so 128 bits in total. The starting values, written as bytes from low-order to high-order, are:

| Word | Byte 1 | Byte 2 | Byte 3 | Byte 4 | As a Hex Number |
|------|--------|--------|--------|--------|-----------------|
| A    | 01     | 23     | 45     | 67     | `67452301`      |
| B    | 89     | AB     | CD     | EF     | `EFCDAB89`      |
| C    | FE     | DC     | BA     | 98     | `98BADCFE`      |
| D    | 76     | 54     | 32     | 10     | `10325476`      |

Together the bytes simply count `01 23 45 ... EF`, then back down `FE DC ... 10`.

#### **Step 4: Process Each Block**

MD5 uses four **auxiliary functions**. Each takes **three 32-bit words** and gives **one 32-bit word**, using bitwise **AND** (written $XY$), **OR** ($\lor$), **NOT** ($\lnot$) and **XOR** ($\oplus$).

| Function | Definition | Used In |
|----------|------------|---------|
| $F(X, Y, Z)$ | $XY \lor \lnot X Z$ | Round 1 |
| $G(X, Y, Z)$ | $XZ \lor Y \lnot Z$ | Round 2 |
| $H(X, Y, Z)$ | $X \oplus Y \oplus Z$ | Round 3 |
| $I(X, Y, Z)$ | $Y \oplus (X \lor \lnot Z)$ | Round 4 |

$F$ works like "**if** $X$ **then** $Y$ **else** $Z$" for each bit.

- **Processing One Block**:
  - **4 rounds**, each with **16 steps**, so **64 steps** in total
  - Each round uses a different auxiliary function ($F$, $G$, $H$, $I$)
  - Each step mixes one of the 16 message words with the buffer, one of the functions, and a constant
  - **64 constants** $T[i] = \lfloor 2^{32} \times |\sin(i)| \rfloor$ for $i = 1, \ldots, 64$, another set of "nothing up my sleeve" numbers
  - Each step also **rotates** bits left by a fixed amount
  - After all 64 steps, the result is **added** to the buffer's value from before the block (mod $2^{32}$)

#### **Step 5: Output**

The final digest is the buffer $a, b, c, d$, starting with $a$ and ending with $d$, with each word written **low-order byte first**: **4 × 32 = 128 bits**.

| Input | MD5 Digest |
|-------|------------|
| (empty) | `d41d8cd98f00b204e9800998ecf8427e` |
| `abc` | `900150983cd24fb0d6963f7d28e17f72` |
| `GeeksForGeeks` | `e39b9c178b2c9be4e99b141d956c6ff6` |

#### **Security of MD5**

- **Broken**:
  - **Collisions** were found in **2004**, and can now be generated in **seconds** on a normal computer
  - In 2012, the **Flame** malware used an MD5 collision to forge a Microsoft signature
  - **Never** use MD5 for passwords, digital signatures or certificates
- **Still Used For**:
  - Non-security **checksums**, such as checking a download wasn't corrupted by accident



In [1]:
# MD5 in Python.
# Part 1: the GeeksforGeeks Java example, using Python's built-in hashlib.
# Part 2: MD5 written from scratch, following the steps in the notes (RFC 1321).
# MD5 is BROKEN for security: use it only as a checksum, never for passwords or signatures.
import hashlib
import math
import os
import struct


# ---------- Part 1: hashlib ----------
def get_md5(text):
    return hashlib.md5(text.encode("utf-8")).hexdigest()


print("Your HashCode Generated by MD5 is:", get_md5("GeeksForGeeks"))


# ---------- Part 2: MD5 from scratch ----------
MASK = 0xFFFFFFFF

# Step 3: the four-word buffer. Written in hex as 67452301 etc.,
# but stored low-order byte first: word A = 01 23 45 67.
IV = [0x67452301, 0xEFCDAB89, 0x98BADCFE, 0x10325476]

# The four auxiliary functions, each taking three 32-bit words
F = lambda x, y, z: (x & y) | (~x & z)          # XY v not(X)Z
G = lambda x, y, z: (x & z) | (y & ~z)          # XZ v Y not(Z)
H = lambda x, y, z: x ^ y ^ z                   # X xor Y xor Z
I = lambda x, y, z: y ^ (x | ~z)                # Y xor (X v not(Z))

# 64 constants T[i] = floor(2^32 * |sin(i)|), and the rotation amounts for each step
T = [int(abs(math.sin(i + 1)) * 2**32) & MASK for i in range(64)]
SHIFTS = [7, 12, 17, 22] * 4 + [5, 9, 14, 20] * 4 + [4, 11, 16, 23] * 4 + [6, 10, 15, 21] * 4


def rotl(x, n):
    return ((x << n) | (x >> (32 - n))) & MASK


def pad(message):
    """Steps 1 and 2: a 1 bit, 0 bits up to 448 mod 512, then the 64-bit length (low byte first)."""
    padded = message + b"\x80"
    while (len(padded) * 8) % 512 != 448:
        padded += b"\x00"
    return padded + struct.pack("<Q", (len(message) * 8) & 0xFFFFFFFFFFFFFFFF)


def md5(message, trace=False):
    padded = pad(message)
    a0, b0, c0, d0 = IV
    for offset in range(0, len(padded), 64):              # each 512-bit block
        X = struct.unpack("<16I", padded[offset:offset + 64])   # 16 words of 32 bits
        a, b, c, d = a0, b0, c0, d0
        for i in range(64):                               # 4 rounds x 16 steps
            if i < 16:
                f, k = F(b, c, d), i                      # round 1
            elif i < 32:
                f, k = G(b, c, d), (5 * i + 1) % 16       # round 2
            elif i < 48:
                f, k = H(b, c, d), (3 * i + 5) % 16       # round 3
            else:
                f, k = I(b, c, d), (7 * i) % 16           # round 4
            new_b = (b + rotl((a + f + T[i] + X[k]) & MASK, SHIFTS[i])) & MASK
            a, b, c, d = d, new_b, b, c
            if trace and i % 16 == 15:
                print(f"  After round {i // 16 + 1}: a={a:08x} b={b:08x} c={c:08x} d={d:08x}")
        a0, b0, c0, d0 = [(x + y) & MASK for x, y in zip([a0, b0, c0, d0], [a, b, c, d])]
    # Step 5: output a, b, c, d, each written low-order byte first
    return struct.pack("<4I", a0, b0, c0, d0).hex()


print("\nPadding:")
for n in [3, 56, 60]:
    print(f"  {n * 8} bits -> {len(pad(b'x' * n)) * 8} bits")

print("\nMD5 of 'abc':")
mine = md5(b"abc", trace=True)
print("  From scratch:", mine)
print("  hashlib:     ", hashlib.md5(b"abc").hexdigest())

tests = {b"": "d41d8cd98f00b204e9800998ecf8427e", b"abc": "900150983cd24fb0d6963f7d28e17f72"}
print("\nRFC 1321 test vectors pass:", all(md5(m) == h for m, h in tests.items()))
print("Matches hashlib on random messages:",
      all(md5(m) == hashlib.md5(m).hexdigest() for m in (os.urandom(n) for n in range(0, 200, 3))))

Your HashCode Generated by MD5 is: e39b9c178b2c9be4e99b141d956c6ff6

Padding:
  24 bits -> 512 bits
  448 bits -> 1024 bits
  480 bits -> 1024 bits

MD5 of 'abc':
  After round 1: a=dda9b9a6 b=72aff2e0 c=e396856b d=a1051895
  After round 2: a=790a77fb b=726342d3 c=a62884aa d=9f47e4f8
  After round 3: a=7f2e507b b=7beb9700 c=8fae6399 d=99d9679d
  After round 4: a=310ade8f b=c08226b3 c=e484b9d8 d=624d8cb2
  From scratch: 900150983cd24fb0d6963f7d28e17f72
  hashlib:      900150983cd24fb0d6963f7d28e17f72

RFC 1321 test vectors pass: True
Matches hashlib on random messages: True


<br>

***

### **Message Authentication Codes (MAC)**

A **Message Authentication Code (MAC)** is another way to check a message's **integrity** and **authenticity**. It's like a hash function, but it also takes a **secret key** $K$. The sender computes a short **tag** from the key and the message, and the receiver, who has the **same key**, recomputes it to check.

$$\text{MAC} = C(K, M)$$

A MAC is also called a **cryptographic checksum**: a small, fixed-size value attached to the message.

| Feature | Hash Function | MAC | Digital Signature |
|---------|---------------|-----|-------------------|
| Key | None | **Shared secret key** | Private / public key pair |
| Who can create it | Anyone | Only key holders | Only the private key holder |
| Who can verify it | Anyone | Only key holders | Anyone with the public key |
| Proves integrity on its own | No, anyone can recompute it | Yes | Yes |
| Non-repudiation | No | **No**, both sides share the key | Yes |

#### **Ways to Use a MAC**

Here $\parallel$ means joining values together, $C$ is the MAC function, and $E$ and $D$ are encryption and decryption.

**(a) Authentication only**

```mermaid
flowchart LR
    M["M"] --> J(("||"))
    M --> C["C with K"] --> J
    J --> T["M || C(K, M)"]
    T --> C2["C with K"] --> CMP{"Compare"}
    T --> CMP
```

- The sender computes $C(K, M)$ and attaches it to $M$
- The receiver computes $C(K, M)$ themselves and compares
- The message is **not** secret

**(b) Authentication and confidentiality, MAC on the plaintext**

```mermaid
flowchart LR
    M["M"] --> J(("||"))
    M --> C["C with K1"] --> J
    J --> E["E with K2"] --> T["E(K2, [M || C(K1, M)])"] --> D["D with K2"] --> R["M and its MAC"]
    R --> C2["C with K1"] --> CMP{"Compare"}
    R --> CMP
```

- Two keys: $K_1$ for the MAC and $K_2$ for encryption
- Compute the MAC on $M$, attach it, then encrypt **both**
- The receiver decrypts first, then checks the MAC

**(c) Authentication and confidentiality, MAC on the ciphertext**

```mermaid
flowchart LR
    M["M"] --> E["E with K2"] --> J(("||"))
    E --> C["C with K1"] --> J
    J --> T["E(K2, M) || C(K1, E(K2, M))"]
    T --> C2["C with K1<br>on the ciphertext"] --> CMP{"Compare"}
    T --> CMP
    CMP -- "Match" --> D["D with K2"] --> P["M"]
```

- Encrypt $M$ first, then compute the MAC on the **ciphertext**
- The receiver checks the MAC **before** decrypting, so tampered messages are rejected without ever being decrypted
- This order, called **encrypt-then-MAC**, is the recommended approach

| Method | Sent | Confidentiality | Authentication Tied To |
|--------|------|-----------------|------------------------|
| (a) | $M \parallel C(K, M)$ | No | Plaintext |
| (b) | $E(K_2, [M \parallel C(K_1, M)])$ | Yes | Plaintext |
| (c) | $E(K_2, M) \parallel C(K_1, E(K_2, M))$ | Yes | Ciphertext |

#### **What a Matching MAC Proves**

- **Not altered**: an attacker who changes the message can't produce a matching MAC without the key
- **From a key holder**: only the sender and receiver know the key, so the message came from one of them
  - The receiver can't **prove** to anyone else who sent it, since they could have made the MAC themselves. A MAC gives **no non-repudiation**
- **In the right order**: if each message includes a **sequence number**, an attacker can't reorder messages without breaking the MAC

A MAC **doesn't need to be reversible**, unlike encryption. Many different messages map to the same tag, and that's fine. MACs are often combined with encryption to get both **confidentiality** and **integrity**.

#### **When MACs Are Useful**

- **Broadcast messages**: when integrity matters but secrecy doesn't, one authority can attach a MAC and every receiver can check it
- **Heavy load**: a busy receiver can check MACs on a **random sample** of messages, which is cheaper than decrypting everything
- **Program integrity**: a program can carry a MAC that's checked only when needed, rather than checking authenticity every time
- **Protocols that separate authentication from confidentiality**, such as **SNMPv3**: commands must be authenticated, but don't need to be secret
- **Different network layers**: authentication and confidentiality can be handled at different layers of the network stack
- **Longer-term protection**: a MAC can be stored with the message and re-checked later. Encrypted data, once decrypted, has no protection left

#### **HMAC: MACs Based on Hash Functions**

Hash-based MACs became popular because hash functions are **fast** and available in almost every programming library. **HMAC** was designed to meet these goals:

- Use an existing, secure hash function **without changing it**
- Allow the hash function to be **swapped** for a better one later
- Keep the hash function's **speed**, with no serious slowdown
- Handle keys **simply and securely**
- Have a **well-understood security proof**

$$\text{HMAC}(K, M) = H\left[(K^+ \oplus \text{opad}) \parallel H\left[(K^+ \oplus \text{ipad}) \parallel M\right]\right]$$

```mermaid
flowchart LR
    K["K"] --> KP["Pad with zeros<br>to b bits = K+"]
    KP --> X1(("⊕ ipad")) --> SI["Si"] --> J1(("||"))
    MSG["M"] --> J1 --> H1["H"] --> J2(("||"))
    KP --> X2(("⊕ opad")) --> SO["So"] --> J2
    J2 --> H2["H"] --> OUT["HMAC(K, M)"]
```

- **HMAC Steps** ($b$ = the hash's block size, e.g. 512 bits for SHA-256):
  - Step 1: Add zeros to the **end** of $K$ to make a $b$-bit string $K^+$. A key longer than $b$ is hashed first
  - Step 2: $S_i = K^+ \oplus \text{ipad}$, where ipad = `00110110` (`0x36`) repeated $b/8$ times
  - Step 3: Append $M$ to $S_i$
  - Step 4: Hash it: $H(S_i \parallel M)$
  - Step 5: $S_o = K^+ \oplus \text{opad}$, where opad = `01011100` (`0x5C`) repeated $b/8$ times
  - Step 6: Append the inner hash to $S_o$
  - Step 7: Hash again to get the HMAC

- **Why Not Just $H(K \parallel M)$?**:
  - SHA-1, SHA-2 and MD5 are vulnerable to **length extension**: knowing $H(K \parallel M)$ lets an attacker compute $H(K \parallel M \parallel \text{extra})$ **without knowing $K$**
  - HMAC's second, outer hash stops this
  - Use **HMAC-SHA256** in practice

#### **DAA: Data Authentication Algorithm**

The **DAA** was based on **DES** in **CBC mode** with an IV of 0. It's **no longer used**. The message is split into 64-bit blocks $D_1, \ldots, D_N$, with the last one padded with 0s.

```mermaid
flowchart LR
    D1["D₁"] --> E1["DES with K"] --> O1["O₁"]
    O1 --> X2(("⊕"))
    D2["D₂"] --> X2 --> E2["DES with K"] --> O2["O₂"] --> DOTS["..."] --> XN(("⊕"))
    DN["D_N"] --> XN --> EN["DES with K"] --> ON["O_N"] --> DAC["DAC = leftmost M bits<br>(16 ≤ M ≤ 64)"]
```

$$
\begin{aligned}
O_1 &= E(K, D_1) \\
O_2 &= E(K, [D_2 \oplus O_1]) \\
O_3 &= E(K, [D_3 \oplus O_2]) \\
&\ \ \vdots \\
O_N &= E(K, [D_N \oplus O_{N-1}])
\end{aligned}
$$

The **Data Authentication Code (DAC)** is all of $O_N$, or its leftmost $M$ bits, where $16 \le M \le 64$.

- **Weaknesses**:
  - DES's **56-bit key** is too short
  - Insecure for messages of **different lengths**: an attacker who knows the MACs of two messages can build a **third message** with a valid MAC

#### **CMAC: Cipher-Based MAC**

**CMAC** fixes DAA's problem with variable-length messages and works with modern block ciphers such as **AES** ($b = 128$) and **3DES** ($b = 64$). It's a NIST standard (SP 800-38B).

| Block Cipher | Block Size $b$ | Key Sizes |
|--------------|----------------|-----------|
| AES | 128 bits | 128, 192 or 256 bits |
| 3DES | 64 bits | 112 or 168 bits |

```mermaid
flowchart LR
    M1["M₁"] --> E1["E with K"] --> C1["C₁"]
    C1 --> X2(("⊕"))
    M2["M₂"] --> X2 --> E2["E with K"] --> C2["C₂"] --> DOTS["..."] --> XN(("⊕"))
    MN["M_n"] --> XN
    K1(["K₁ (or K₂ if padded)"]) --> XN
    XN --> EN["E with K"] --> CN["C_n"] --> T["T = MSB_Tlen(C_n)"]
```

$$
\begin{aligned}
C_1 &= E(K, M_1) \\
C_2 &= E(K, [M_2 \oplus C_1]) \\
&\ \ \vdots \\
C_n &= E(K, [M_n \oplus C_{n-1} \oplus K_1]) \\
T &= \text{MSB}_{T_{len}}(C_n)
\end{aligned}
$$

- $T_{len}$: the length of the tag $T$ in bits
- $\text{MSB}_{T_{len}}$: the leftmost $T_{len}$ bits

- **The Subkeys $K_1$ and $K_2$**:
  - Both are **derived from $K$**: compute $L = E(K, 0^b)$, then $K_1 = L \cdot x$ and $K_2 = L \cdot x^2$ in $GF(2^b)$
  - **$K_1$** is used if the last block is **complete**
  - **$K_2$** is used if the last block had to be **padded** (with a 1 bit, then 0s)
  - Mixing a secret subkey into the last block is what stops the variable-length attack on DAA

| MAC | Built From | Status |
|-----|------------|--------|
| DAA | DES in CBC mode | **Withdrawn**: short key, insecure for variable-length messages |
| CMAC | AES or 3DES | Secure, NIST standard |
| HMAC | Any hash function (e.g. SHA-256) | Secure, the most widely used MAC |

<br>

***

### **Digital Signatures**

**Digital Signatures** are one of the most important results of public-key cryptography. Earlier techniques focused on stopping **outsiders** from reading or changing messages. But they couldn't settle disputes **between the two parties themselves**, such as one of them backing out of an agreement and claiming they never sent a message.

A digital signature is like signing your name on paper: a **mark that only you can make**, attached to a document. It guarantees:

- **Authentication**: who sent the message
- **Integrity**: the message hasn't been changed
- **Non-repudiation**: the sender can't later deny sending it

Digital signatures let parties who **don't fully trust each other** communicate safely. They also allow:

- **Verification of the author**, and of the date and time, if a timestamp is included in what's signed
- **Authentication of the contents** at the moment of signing
- **Third-party verification**, so a judge or arbitrator can settle disputes

#### **Why MACs Aren't Enough**

With a MAC, Alice and Bob **share the same secret key**, which causes two problems:

- **Alice could forge** a message, compute its MAC, and claim Bob sent it
- **Bob could deny** sending a real message, claiming Alice made it up

Since both could have created the MAC, no one can tell who's telling the truth. With a digital signature, **only Bob has his private key**, so only Bob could have signed.

#### **How a Digital Signature Works**

```mermaid
flowchart LR
    subgraph BOB["Bob (signer)"]
        direction TB
        M1["Message M"] --> H1["Hash function"] --> h1["h"] --> E["Sign with<br>Bob's private key"] --> S["Signature S"]
    end
    subgraph ALICE["Alice (verifier)"]
        direction TB
        M2["Message M"] --> H2["Hash function"] --> h2["h"] --> CMP{"Compare"}
        S2["Signature S"] --> D["Check with<br>Bob's public key"] --> h3["h′"] --> CMP
        CMP --> R["Valid / not valid"]
    end
    BOB -- "M and S" --> ALICE
```

- **Bob Signs**:
  - Hashes the message $M$ to get a short, fixed-size digest $h$
  - Signs $h$ with his **private key** to get the signature $S$. With RSA, this means "encrypting" $h$ with the private key
  - Optionally, encrypts $M$ and $S$ with **Alice's public key** for confidentiality

- **Alice Verifies**:
  - Decrypts with her private key, if Bob encrypted the message
  - Hashes $M$ herself to get $h$
  - Uses **Bob's public key** on the signature to recover $h'$
  - If $h = h'$, the message is **unchanged** and **came from Bob**, and Bob **can't deny** signing it

This relies on Bob's public key **really belonging to Bob**, which is the job of **certificates**, and on his private key not being stolen.

#### **Requirements for a Digital Signature**

A digital signature is a **concept** rather than a single algorithm. Any scheme must meet these requirements:

- The signature must be a bit pattern that **depends on the message** being signed
- It must use information **unique to the sender**, so it can't be forged or denied
- It must be **easy to produce**
- It must be **easy to verify**
- **Forging must be infeasible**: both making a new message for an existing signature, and making a fake signature for a given message
- It must be practical to **store a copy** of the signature

#### **ElGamal Digital Signature Scheme**

Based on the **ElGamal cryptosystem** and the discrete logarithm problem. It uses a prime $q$ and a primitive root $\alpha$ of $q$.

**Key generation:**

$$1 < X_A < q - 1 \text{ (random)} \qquad Y_A = \alpha^{X_A} \bmod q$$

$$\text{Private key} = X_A \qquad \text{Public key} = \{q, \alpha, Y_A\}$$

**Signing** message $M$:

- Compute $m = H(M)$, with $0 \le m \le q - 1$
- Choose a random $K$ with $1 \le K \le q - 1$ and $\gcd(K, q - 1) = 1$
- $S_1 = \alpha^K \bmod q$
- Compute $K^{-1} \bmod (q - 1)$
- $S_2 = K^{-1}(m - X_A S_1) \bmod (q - 1)$
- The signature is $(S_1, S_2)$

**Verifying:**

$$V_1 = \alpha^m \bmod q \qquad V_2 = (Y_A)^{S_1}(S_1)^{S_2} \bmod q$$

The signature is valid **only if $V_1 = V_2$**.

#### **ElGamal Signature Execution for $q = 19$, $\alpha = 10$**

The primitive roots of 19 are $\{2, 3, 10, 13, 14, 15\}$.

| Step | Who | Working | Result |
|------|-----|---------|--------|
| Private key | Alice | Chooses $X_A$ | $16$ |
| Public key | Alice | $Y_A = 10^{16} \bmod 19$ | $4$, so $\{19, 10, 4\}$ |
| Hash | Alice | $m = H(M)$ | $14$ |
| Random $K$ | Alice | $\gcd(5, 18) = 1$ | $K = 5$ |
| $S_1$ | Alice | $10^5 \bmod 19$ | $3$ |
| $K^{-1}$ | Alice | $5 \times 11 = 55 = 3 \times 18 + 1$ | $11$ |
| $S_2$ | Alice | $11 \times (14 - 16 \times 3) \bmod 18 = -374 \bmod 18$ | $4$ |
| $V_1$ | Bob | $10^{14} \bmod 19$ | $16$ |
| $V_2$ | Bob | $4^3 \times 3^4 \bmod 19 = 7 \times 5 \bmod 19$ | $16$ |

Final result:
- Signature $(S_1, S_2) = (3, 4)$
- $V_1 = V_2 = 16$, so the signature is **valid** ✓
- **Never reuse $K$**: two signatures made with the same $K$ let an attacker work out $K$, and then the **private key**

#### **Schnorr Digital Signature Scheme**

Also based on the **discrete logarithm problem**. Its advantage is that most of the work is **message-independent**, so it can be done **in advance** when the CPU is idle. The message-dependent part is just a quick multiplication of a $2n$-bit number by an $n$-bit number.

**Key generation:**

- Choose primes $p$ and $q$, where $q$ is a **prime factor of $p - 1$**
- Choose $a$ with $a^q \equiv 1 \pmod{p}$ (and $a \neq 1$)
- $\{a, p, q\}$ is the **global public key**, which can be shared by a group of users
- **Private key**: random $s$ with $0 < s < q$
- **Public key**: $v = a^{-s} \bmod p$

**Signing:**

- Choose a random $r$ with $0 < r < q$, and compute $x = a^r \bmod p$ (this can be done in advance)
- $e = H(M \parallel x)$
- $y = (r + se) \bmod q$
- The signature is $(e, y)$

**Verifying:**

- $x' = a^y v^e \bmod p$
- Valid if $e = H(M \parallel x')$

This works because $a^y v^e = a^{r + se} \cdot a^{-se} = a^r = x$.

#### **Digital Signature Standard (DSS)**

The **DSS** is a NIST standard. It originally used the **Digital Signature Algorithm (DSA)** with SHA, and later versions added **RSA** and **elliptic curve (ECDSA, EdDSA)** signatures.

Unlike RSA, DSA can **only sign**. It can't be used for encryption. The signer needs a key pair, a hash of the message, a **random per-message value $k$**, and **global parameters** shared by everyone using the scheme.

```mermaid
flowchart LR
    subgraph SIGN["Signing"]
        direction TB
        M["M"] --> H["H"] --> SIG["Sign"]
        K1(["k (random)"]) --> SIG
        PR(["Private key x"]) --> SIG
        G1(["Global p, q, g"]) --> SIG
        SIG --> RS["Signature (r, s)"]
    end
    subgraph VER["Verifying"]
        direction TB
        M2["M′"] --> H2["H"] --> V["Verify"]
        RS2["(r′, s′)"] --> V
        PU(["Public key y"]) --> V
        G2(["Global p, q, g"]) --> V
        V --> CMP{"v = r′ ?"}
    end
    SIGN -- "M, r, s" --> VER
```

**Global public key components:**

- $p$: a prime with $2^{L-1} < p < 2^L$, where $L$ is a multiple of 64 and $512 \le L \le 1024$
- $q$: a prime divisor of $p - 1$, with $2^{159} < q < 2^{160}$
- $g = h^{(p-1)/q} \bmod p$, for any $1 < h < p - 1$ that makes $g > 1$

**User keys:**

- **Private key**: $x$ with $0 < x < q$
- **Public key**: $y = g^x \bmod p$
- **Per-message secret**: a new random $k$ with $0 < k < q$ for every signature

**Signing:**

$$r = (g^k \bmod p) \bmod q$$

$$s = [k^{-1}(H(M) + xr)] \bmod q$$

The signature is $(r, s)$.

**Verifying** ($M'$, $r'$, $s'$ are the received values):

$$w = (s')^{-1} \bmod q$$

$$u_1 = [H(M')\,w] \bmod q \qquad u_2 = (r')\,w \bmod q$$

$$v = [(g^{u_1} y^{u_2}) \bmod p] \bmod q$$

The signature is valid if $v = r'$.

- **DSA Notes**:
  - Like ElGamal, **reusing $k$ reveals the private key**. Sony's PlayStation 3 signing key was leaked this way
  - The key sizes above are from the **original** standard. Later versions allow larger $p$ and $q$
  - The current standard (**FIPS 186-5**) no longer approves DSA for creating new signatures. **ECDSA**, **EdDSA** and **RSA** are used instead

#### **Comparison of Signature Schemes**

| Scheme | Hard Problem | Signature | Notes |
|--------|--------------|-----------|-------|
| RSA | Integer factorisation | One number | Can also encrypt |
| ElGamal | Discrete logarithm | $(S_1, S_2)$ | Random $K$ must never be reused |
| Schnorr | Discrete logarithm | $(e, y)$ | Most work done in advance |
| DSA | Discrete logarithm | $(r, s)$ | Signing only, being phased out |
| ECDSA / EdDSA | Elliptic curve discrete log | Two numbers | Small keys, widely used today |

<br>

***

### **Advanced Digital Signatures**

Standard digital signatures involve **one signer** and **one message**. Advanced signature schemes handle **groups of signers**, **many signatures at once**, or **anonymity**. This lesson covers **five** of them.

| Scheme | Who Signs | Verifier Learns Who Signed? | Manager? | Main Idea |
|--------|-----------|-----------------------------|----------|-----------|
| **Threshold** | Any $t$ of $n$ members together | No, it looks like one signature | No (or a one-time dealer) | Needs enough members to agree |
| **Aggregate** | Many signers, different messages | Yes | No | Many signatures squeezed into one |
| **Group** | One member, on behalf of the group | No, but the manager can find out | **Yes** | Anonymous, but traceable |
| **Ring** | One member of an ad hoc "ring" | No, and **no one** can find out | No | Anonymous, untraceable |
| **Multi-signature** | Several known signers, same message | Yes | No | Compact joint signature |

#### **Threshold Signature Scheme**

A group of $n$ members shares one signing key, but signing needs **at least $t$ of them** to cooperate. This is called a **$t$-of-$n$** scheme. Fewer than $t$ members **can't** sign.

```mermaid
flowchart LR
    K(["Group private key<br>(never in one place)"]) -.-> S1["Share 1"] & S2["Share 2"] & S3["Share 3"] & S4["Share 4"] & S5["Share 5"]
    S1 --> SIGN["Any 3 shares<br>sign together"]
    S3 --> SIGN
    S4 --> SIGN
    SIGN --> SIG["One normal-looking signature"] --> V["Verified with the<br>group public key"]
```

- **How It Works**:
  - The private key is split into **secret shares**, one per member, and no member knows anyone else's share
  - The **full private key is never stored in one place**: it's either split once by a trusted dealer, or created jointly with no one ever seeing it (**distributed key generation**)
  - Any $t$ members combine their shares to sign. With fewer than $t$, signing is impossible
  - The result looks like an **ordinary signature**, so the verifier doesn't learn **which** members took part
  - Members can be chosen from a larger set of possible signers

- **Why It's Useful**:
  - **No single point of failure**: stealing one share isn't enough
  - **No single person** can sign alone
  - Used for company approvals, certificate authorities and cryptocurrency wallets

#### **Aggregate Signature Scheme**

An **aggregate signature** combines $n$ signatures, from **different signers on different messages**, into **one short signature**, the same size as a single signature.

| Algorithm | Input | Output |
|-----------|-------|--------|
| **KeyGen** | - | Key pair $(sk_i, pk_i)$ |
| **Sign** | Message $m_i$, secret key $sk_i$ | Signature $\sigma_i$ |
| **Combine** | $n$ triples $(m_i, pk_i, \sigma_i)$ | One aggregate signature $\sigma$ |
| **Verify** | $n$ pairs $(pk_i, m_i)$ and $\sigma$ | True only if **all $n$** original signatures were valid |

Standard signature schemes have KeyGen, Sign and Verify. Aggregate schemes add **Combine**.

- **Why It's Useful**:
  - Saves space and bandwidth: 1,000 signatures can be stored as one
  - Example: **BLS signatures**, used in blockchains, where thousands of validators each sign

#### **Group Signature Scheme**

A member of a group can sign **on behalf of the group**. Verifiers can check that **some member** signed, but not **which** one. A trusted **group manager** can **reveal the signer** if there's a dispute.

- **Operations**:
  - **Setup**: create the group manager's key pair (a group public key and a manager private key)
  - **Join / Leave**: add or remove members, each with their own signing key
  - **Sign**: a member signs with their own private key
  - **Verify**: anyone checks the signature using the **group public key**
  - **Open**: the manager uses their private key to **identify the signer**

- **Static vs Dynamic Groups**:
  - **Static**: members are fixed when the group is set up. Simpler, but members can't be added later
  - **Dynamic**: members can **join and leave** over time. More flexible, but removing members (revocation) is harder

- **Example Use**: an employee signs on behalf of a company. Customers only see "signed by Company X", but the company can find out which employee signed if something goes wrong

#### **Ring Signature Scheme**

A **ring signature** is like a group signature, but with **no manager at all**. The signer picks **any set of public keys** (the "ring"), including their own, and signs. The name comes from the ring-shaped structure of the equation used.

- **How It Differs from Group Signatures**:
  - **No setup or management**: the ring is formed on the spot, and the other members don't even need to know they're in it
  - **No one can reveal the signer**, since there's no manager
  - Basic ring signatures are **unlinkable**: you can't tell if two signatures came from the same person
  - **Linkable ring signatures** are a variant that can spot when the same person signs twice, while still hiding who they are

- **Example Uses**:
  - **Whistleblowing**: proving a leak came from "one of these senior officials", without revealing which one
  - **Privacy coins** such as Monero

#### **Multi-Signature Scheme**

In a **multi-signature** scheme, **several known signers** all sign the **same message**, and the result is **one compact signature**, instead of a list of separate signatures.

- **How It Differs**:
  - **Not anonymous**: the verifier knows exactly which public keys signed
  - All the signers sign the **same** message (aggregate signatures allow **different** messages)
  - Every required signer must take part (unlike threshold, where any $t$ of $n$ is enough)
  - Example: a contract signed by all company directors, or a shared cryptocurrency account

| Feature | Threshold | Multi-Signature | Aggregate |
|---------|-----------|-----------------|-----------|
| Signers | Any $t$ of $n$ | All required signers | Any number |
| Messages | One | One | Different messages |
| Verifier sees who signed | No | Yes | Yes |
| Result | One signature | One compact signature | One compact signature |

<br>

***